In [1]:
!pip install pandas openpyxl
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset


In [2]:
# ===== Paramètres =====
INPUT_FILE  = "/Users/y-kouadio/Documents/DATA/AOUT PLATEFORME.xlsx"     # <-- mets le nom/chemin de ton fichier
SHEET_NAME  = "AOUT PLATEFORME"
OUTPUT_FILE = "global_database_processed.xlsx"

In [3]:
# (optionnel) voir les feuilles disponibles
print(pd.ExcelFile(INPUT_FILE, engine="openpyxl").sheet_names)


['AOUT PLATEFORME', 'Extraction_Grouped']


In [4]:
# ===== 0) Lecture =====
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, engine="openpyxl")


In [13]:
# ===== 1) Références de colonnes par position (comme dans ton VBA) =====
cols = []
if len(df.columns) >= 5:  cols.append(df.columns[4])    # 5ème
if len(df.columns) >= 38: cols.append(df.columns[37])   # 38ème

if cols:
    display(df[cols].head(10))  # montre les 10 premières lignes de ces colonnes
else:
    print("Aucune des colonnes demandées n'est disponible.")


,PRODUIT,FGA
0,MOTO,125.0
1,MOTO,125.0
2,MOTO,125.0
3,MOTO,125.0
4,SUR_MESURE,887.0
5,MOTO,125.0
6,MOTO,125.0
7,MOTO,125.0
8,MOTO,125.0
9,MOTO,125.0


In [ ]:
# ===== 2) Date d’effet (LEFT 10, remplacement '-' -> '/', test IsDate) =====
raw_10 = (
    df[COL5_EFFECT_RAW].astype(str).str.slice(0, 10).str.replace("-", "/", regex=False)
)

In [6]:
from IPython.display import display

# 1) Vérifier / afficher les colonnes extraites (noms)
print("Colonnes demandées (1-based) :", [13, 5, 39, 11, 2, 26, 12, 6, 31])
cols_to_extract_idx1 = [13, 5, 39, 11, 2, 26, 12, 6, 31]
cols_to_extract = [df.columns[i-1] for i in cols_to_extract_idx1]
print("→ Noms des colonnes extraites :", cols_to_extract)



Colonnes demandées (1-based) : [13, 5, 39, 11, 2, 26, 12, 6, 31]
→ Noms des colonnes extraites : ['NOM COMPLET', 'PRODUIT', 'PRIME TTC', 'IMMATRICULATION', 'TELEPHONE', 'DUREE ( MOIS)', 'DATE ECHEANCE', 'quoteNo', 'EMAIL AGENT']


In [7]:
 # 2) Construire le DataFrame d’extraction
extraction = df[cols_to_extract].copy()

In [8]:
# 3) Donner la taille et un aperçu
print("\nTaille de l’extraction (lignes, colonnes) :", extraction.shape)
display(extraction.head(20))   # affiche les 20 premières lignes


Taille de l’extraction (lignes, colonnes) : (19879, 9)


,NOM COMPLET,PRODUIT,PRIME TTC,IMMATRICULATION,TELEPHONE,DUREE ( MOIS),DATE ECHEANCE,quoteNo,EMAIL AGENT
0,PEHE SOUEHI JEAN-PAUL,MOTO,8580,4877LT04,0777084048,12.000000,2025-08-20,SNE32E342B2024,hamidouzongo92@gmail.com
1,SIELOU HOUSSOU TANOH KOUAME NOE,MOTO,8580,6467JL01,0708703135,12.000000,2025-08-09,SN860E64212024,lalayoyo12@yahoo.fr
2,KONAN OI KONAN EVARISTE,MOTO,8580,6485JS04,0779491211,12.000000,2025-08-19,SN9F902B2A2024,ozidjezago10@gmail.com
3,BAMBA TILONON DAMIEN,MOTO,8580,8827JV04,0505465065,12.166667,2025-08-06,SN426BB4E32024,2005quaddy@gmail.com
4,DIAKITE SIAKA ADAM,SUR_MESURE,82950,8137JE01,0707376230,12.166667,2025-08-23,SN8AEE4B7F2024,senfousenfou10@gmail.com
5,DIOMANDE MOUSSA,MOTO,8580,7708LZ03,0707540064,12.166667,2025-08-03,SN5C6A25832024,loukouloukou634@gmail.com
6,DIABATE AMADOU,MOTO,8580,6892LN01,0707518193,12.000000,2025-08-25,SNACD011332024,syllasiriki2015@gmail.com
7,DEA SERGE PACÔME,MOTO,8580,5003KU01,0747414314,12.166667,2025-08-10,SN023EBADF2024,patrickange622@gmail.com
8,KONE YAYA,MOTO,8580,CH04744,0709480396,12.166667,2025-08-16,SN8F1FB7262024,cmamadou86@gmail.com
9,OUATTARA HIKANIN,MOTO,8580,9232LG03,0747730864,12.000000,2025-08-02,SNFB7E88822024,mohamedbilla2015@gmail.com


In [9]:

# 4) (Optionnel) aperçu des valeurs manquantes par colonne
print("\nValeurs manquantes par colonne :")
print(extraction.isna().sum())


Valeurs manquantes par colonne :
NOM COMPLET            0
PRODUIT                0
PRIME TTC              0
IMMATRICULATION        0
TELEPHONE          10179
DUREE ( MOIS)       3032
DATE ECHEANCE          0
quoteNo                0
EMAIL AGENT            0
dtype: int64


In [10]:
BASE_URL = "https://ci.leadway.com/auto/quote/"

# --- localiser la colonne quoteNo (insensible à la casse) ---
col_candidates = [c for c in extraction.columns if c.lower() == "quoteno"]
if not col_candidates:
    raise KeyError("Colonne 'quoteNo' introuvable dans 'extraction'. Vérifie le nom exact.")
qcol = col_candidates[0]

# --- construire le lien ---
extraction[qcol] = extraction[qcol].astype("string").str.strip()
# garder NaN si quoteNo est vide
mask = extraction[qcol].notna() & (extraction[qcol] != "")
extraction.loc[mask, qcol] = BASE_URL + extraction.loc[mask, qcol]

# --- renommer la colonne ---
extraction.rename(columns={qcol: "LIEN DE RENOUVELLEMENT"}, inplace=True)

# aperçu
print("Colonnes :", list(extraction.columns))
extraction.head(10)


Colonnes : ['NOM COMPLET', 'PRODUIT', 'PRIME TTC', 'IMMATRICULATION', 'TELEPHONE', 'DUREE ( MOIS)', 'DATE ECHEANCE', 'LIEN DE RENOUVELLEMENT', 'EMAIL AGENT']


,NOM COMPLET,PRODUIT,PRIME TTC,IMMATRICULATION,TELEPHONE,DUREE ( MOIS),DATE ECHEANCE,LIEN DE RENOUVELLEMENT,EMAIL AGENT
0,PEHE SOUEHI JEAN-PAUL,MOTO,8580,4877LT04,0777084048,12.000000,2025-08-20,https://ci.leadway.com/auto/quote/SNE32E342B2024,hamidouzongo92@gmail.com
1,SIELOU HOUSSOU TANOH KOUAME NOE,MOTO,8580,6467JL01,0708703135,12.000000,2025-08-09,https://ci.leadway.com/auto/quote/SN860E64212024,lalayoyo12@yahoo.fr
2,KONAN OI KONAN EVARISTE,MOTO,8580,6485JS04,0779491211,12.000000,2025-08-19,https://ci.leadway.com/auto/quote/SN9F902B2A2024,ozidjezago10@gmail.com
3,BAMBA TILONON DAMIEN,MOTO,8580,8827JV04,0505465065,12.166667,2025-08-06,https://ci.leadway.com/auto/quote/SN426BB4E32024,2005quaddy@gmail.com
4,DIAKITE SIAKA ADAM,SUR_MESURE,82950,8137JE01,0707376230,12.166667,2025-08-23,https://ci.leadway.com/auto/quote/SN8AEE4B7F2024,senfousenfou10@gmail.com
5,DIOMANDE MOUSSA,MOTO,8580,7708LZ03,0707540064,12.166667,2025-08-03,https://ci.leadway.com/auto/quote/SN5C6A25832024,loukouloukou634@gmail.com
6,DIABATE AMADOU,MOTO,8580,6892LN01,0707518193,12.000000,2025-08-25,https://ci.leadway.com/auto/quote/SNACD011332024,syllasiriki2015@gmail.com
7,DEA SERGE PACÔME,MOTO,8580,5003KU01,0747414314,12.166667,2025-08-10,https://ci.leadway.com/auto/quote/SN023EBADF2024,patrickange622@gmail.com
8,KONE YAYA,MOTO,8580,CH04744,0709480396,12.166667,2025-08-16,https://ci.leadway.com/auto/quote/SN8F1FB7262024,cmamadou86@gmail.com
9,OUATTARA HIKANIN,MOTO,8580,9232LG03,0747730864,12.000000,2025-08-02,https://ci.leadway.com/auto/quote/SNFB7E88822024,mohamedbilla2015@gmail.com


In [13]:
from pathlib import Path
import os
import pandas as pd

INPUT_FILE  = "/Users/y-kouadio/Documents/DATA/AOUT PLATEFORME.xlsx"
SHEET_NAME  = "AOUT_PLATEFORME"

# -> même répertoire que le fichier source
BASE_DIR    = Path(INPUT_FILE).parent
OUTPUT_FILE = str(BASE_DIR / "global_database_processed.xlsx")

# (exemple d'écriture)
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", date_format="DD/MM/YYYY") as writer:
    df.to_excel(writer, sheet_name=f"{SHEET_NAME}_processed", index=False)
    extraction.to_excel(writer, sheet_name="Extraction", index=False)

print("Enregistré ici :", os.path.abspath(OUTPUT_FILE))


Enregistré ici : /Users/y-kouadio/Documents/DATA/global_database_processed.xlsx


In [14]:
# --- Paramètres à adapter si besoin ---
INPUT_FILE   = "global_database_processed.xlsx"   # fichier qui contient la feuille "Extraction"
SOURCE_SHEET = "Extraction"                       # la feuille créée à l'étape 1
N_COLS       = 9                                  # on copie les 9 premières colonnes (A:I)
EMAIL_COL_1B = 9                                  # l'email est en colonne I (index 1-based VBA)


In [46]:
import pandas as pd

print("Feuilles de INPUT_FILE :")
print(pd.ExcelFile(INPUT_FILE, engine="openpyxl").sheet_names)

print("\nFeuilles de OUTPUT_FILE :")
print(pd.ExcelFile(OUTPUT_FILE, engine="openpyxl").sheet_names)


Feuilles de INPUT_FILE :
['AOUT PLATEFORME']

Feuilles de OUTPUT_FILE :
['AOUT_PLATEFORME_processed', 'Extraction']


In [15]:
SOURCE_FILE  = OUTPUT_FILE
SOURCE_SHEET = "Extraction"

df_src = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET, engine="openpyxl")
df_src.head()


,NOM COMPLET,PRODUIT,PRIME TTC,IMMATRICULATION,TELEPHONE,DUREE ( MOIS),DATE ECHEANCE,LIEN DE RENOUVELLEMENT,EMAIL AGENT
0,PEHE SOUEHI JEAN-PAUL,MOTO,8580,4877LT04,0777084048,12.000000,2025-08-20,https://ci.leadway.com/auto/quote/SNE32E342B2024,hamidouzongo92@gmail.com
1,SIELOU HOUSSOU TANOH KOUAME NOE,MOTO,8580,6467JL01,0708703135,12.000000,2025-08-09,https://ci.leadway.com/auto/quote/SN860E64212024,lalayoyo12@yahoo.fr
2,KONAN OI KONAN EVARISTE,MOTO,8580,6485JS04,0779491211,12.000000,2025-08-19,https://ci.leadway.com/auto/quote/SN9F902B2A2024,ozidjezago10@gmail.com
3,BAMBA TILONON DAMIEN,MOTO,8580,8827JV04,0505465065,12.166667,2025-08-06,https://ci.leadway.com/auto/quote/SN426BB4E32024,2005quaddy@gmail.com
4,DIAKITE SIAKA ADAM,SUR_MESURE,82950,8137JE01,0707376230,12.166667,2025-08-23,https://ci.leadway.com/auto/quote/SN8AEE4B7F2024,senfousenfou10@gmail.com


In [16]:
# 2) Déterminer les colonnes à copier et la colonne e-mail (par position)
cols_to_copy = list(df_src.columns[:N_COLS])      # A..I
email_col    = df_src.columns[EMAIL_COL_1B - 1]   # I

In [17]:
# 3) Écrire un tableau par email, côte à côte
out_sheet = "Extraction_Grouped"                  # nouvelle feuille
with pd.ExcelWriter(INPUT_FILE, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    start_col = 0                                 # on commence en colonne A de la nouvelle feuille
    for email, grp in df_src.groupby(email_col, dropna=False):
        block = grp[cols_to_copy]
        # écrit l’en-tête + lignes, puis laisse une colonne vide entre chaque bloc
        block.to_excel(writer, sheet_name=out_sheet, index=False,
                       startrow=0, startcol=start_col)
        start_col += N_COLS + 1                   # 9 colonnes + 1 vide = comme VBA (K, T, …)

print("✔ Regroupement terminé dans la feuille 'Extraction_Grouped'.")

BadZipFile: Bad CRC-32 for file 'docProps/core.xml'

In [55]:
from openpyxl import load_workbook
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side
from openpyxl.utils import get_column_letter

# ===== Paramètres =====
INPUT_FILE  = "global_database_processed.xlsx"  # le fichier qui contient "Extraction"
SHEET_NAME  = "Extraction"
N_COLS      = 9          # on copie A..I
EMAIL_COL   = 9          # email en colonne I (index 1-based)
START_COL   = 11         # on commence à écrire en K
BLOCK_GAP   = 1          # 1 colonne vide entre blocs
MAX_XLSX_COLS = 16384    # limite Excel (.xlsx) = XFD


In [56]:
# ===== Styles (entête) =====
header_fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")  # orange
header_font = Font(bold=True, color="000000")  # noir, gras
thin = Side(style="thin", color="000000")
border_all = Border(left=thin, right=thin, top=thin, bottom=thin)


In [58]:
# ===== Déterminer la dernière ligne utile de la colonne email =====
lastRow = ws.max_row
while lastRow > 1 and (ws.cell(row=lastRow, column=EMAIL_COL).value in (None, "")):
    lastRow -= 1

# Sécurité : si aucune donnée
if lastRow < 2:
    print("Aucune donnée dans 'Extraction'. Rien à faire.")
else:
    # ===== Construire les groupes d'emails (triés) =====
    email_dict = {}
    for r in range(2, lastRow + 1):
        val = ws.cell(row=r, column=EMAIL_COL).value
        if val in (None, ""):
            continue  # ignore lignes sans email
        key = str(val).strip()
        email_dict.setdefault(key, []).append(r)

In [54]:
# --- Ouvrir le classeur et la feuille ---
wb =load_workbook(OUTPUT_FILE)
ws = wb[SHEET_NAME]


In [60]:
# --- Afficher un résumé des groupes (sans écriture pour tester)
if not email_dict:
    print("Aucun email trouvé dans la colonne I.")
else:
    groups = sorted(email_dict.items(), key=lambda kv: kv[0].lower())
    print("Groupes détectés :", len(groups))
    # top 10 pour aperçu
    for email, rows in groups[:10]:
        print(f"- {email} : {len(rows)} lignes")

Groupes détectés : 1126
- 2005quaddy@gmail.com : 5 lignes
- 2alassurances@gmail.com : 6 lignes
- 2ka.assurances@gmail.com : 41 lignes
- 717fabus@gmail.com : 1 lignes
- _dmas1567@gmail.com : 4 lignes
- _kouakoukoffiroger969@gmail.com : 1 lignes
- a-aja@gmail.com : 16 lignes
- a-ba@gmail.com : 12 lignes
- a-cisse@gmail.com : 66 lignes
- a-ndri@gmail.com : 8 lignes


In [68]:
# À placer juste après la construction de email_dict
groups = sorted(email_dict.items(), key=lambda kv: kv[0].lower())
nb_groups = len(groups)

In [63]:
# ===== Nettoyer l’ancienne zone à droite de K (optionnel mais conseillé) =====
# On efface tout ce qui est à droite de START_COL
max_col_existing = ws.max_column
max_row_existing = ws.max_row

if max_col_existing >= START_COL:
    for r in range(1, max_row_existing + 1):
        for c in range(START_COL, max_col_existing + 1):
            ws.cell(row=r, column=c).value = None


In [65]:
# ===== Calcul de la capacité de blocs par rangée (anti-dépassement XFD) =====
block_width = N_COLS + BLOCK_GAP
usable_cols = MAX_XLSX_COLS - (START_COL - 1)
max_blocks_per_row = max(1, usable_cols // block_width)
print("Capacité max de blocs par rangée :", max_blocks_per_row)


Capacité max de blocs par rangée : 1637


In [1]:
# prérequis: groups, nb_groups, max_blocks_per_row, currentRowStart définis
print("nb_groups =", nb_groups, "| max_blocks_per_row =", max_blocks_per_row)

if nb_groups == 0:
    print("Aucun groupe d'email trouvé.")
else:
    for idx in range(0, nb_groups, max_blocks_per_row):
        band = groups[idx : idx + max_blocks_per_row]

        # ... ici tu écris chaque bloc de la bande ...
        # Exemple minimal pour avancer la hauteur :
        max_height = max((len(rows) for _, rows in band), default=0) + 1
        currentRowStart += max_height + 1

        print(f"Bande traitée: idx={idx} → {min(idx+max_blocks_per_row, nb_groups)-1}")


NameError: name 'nb_groups' is not defined

In [ ]:
# ===== Écriture par bandes (rangées de blocs), puis retour à la ligne =====
    currentRowStart = 1  # on pose les entêtes de la bande ici
    idx = 0
    while idx < nb_groups:
        band = groups[idx : idx + max_blocks_per_row]

        # Écrire chaque bloc de la bande
        band_heights = []
        for j, (email, rows) in enumerate(band):
            destCol = START_COL + j * block_width  # K, K+10, K+20, ...
            # 1) Entêtes A..I -> (currentRowStart, destCol..destCol+N_COLS-1)
            for c in range(1, N_COLS + 1):
                src_val = ws.cell(row=1, column=c).value
                cell = ws.cell(row=currentRowStart, column=destCol + c - 1)
                cell.value = src_val
                cell.fill = header_fill
                cell.font = header_font
                cell.border = border_all


In [ ]:

    
            # 2) Données du groupe
            destRow = currentRowStart + 1
            for srcRow in rows:
                for c in range(1, N_COLS + 1):
                    dst = ws.cell(row=destRow, column=destCol + c - 1)
                    dst.value = ws.cell(row=srcRow, column=c).value
                    dst.border = border_all
                destRow += 1

            band_heights.append(len(rows) + 1)  # +1 pour ligne entête

            # (Optionnel) largeur de colonnes lisible
            for c in range(N_COLS):
                col_letter = get_column_letter(destCol + c)
                # largeur fixe raisonnable; pour un vrai AutoFit on devrait mesurer la longueur des valeurs
                ws.column_dimensions[col_letter].width = 18

        # Descendre pour la bande suivante (+1 ligne vide)
        currentRowStart += (max(band_heights) if band_heights else 0) + 1
        idx += max_blocks_per_row

    wb.save(INPUT_FILE)
    print("✔ Données regroupées par email dans 'Extraction' (à partir de la colonne K), avec retour à la ligne auto.")

In [ ]:






# tentative de parsing robuste (jour d'abord si besoin)
formatted_date = pd.to_datetime(raw_10, errors="coerce", dayfirst=True, infer_datetime_format=True)
df["formatted_date"] = formatted_date            # ≈ ta colonne 97 (formatée)

# ===== 3) Durée en mois = (col 38) / 30 =====
duration_months = pd.to_numeric(df[COL38_DURATION_DAYS], errors="coerce") / 30
df["duration_months"] = duration_months         # ≈ ta colonne 98

# ===== 4) Date d’échéance = date effet + mois(entiers) - 1 jour =====
def compute_due(d, m):
    if pd.isna(d) or pd.isna(m):
        return pd.NaT
    # VBA DateAdd("m", m, ...) se comporte comme des mois entiers.
    m_int = int(np.floor(m))
    return d + DateOffset(months=m_int) - pd.Timedelta(days=1)

df["due_date"] = [compute_due(d, m) for d, m in zip(df["formatted_date"], df["duration_months"])]
# ≈ ta colonne 99 dans VBA

# ===== 5) (ProcessContractData2) Copier LEFT(10) de la col 5 (utile pour debug) =====
df["effect_10chars"] = raw_10


# ===== 7) Sauvegarde dans un nouveau fichier Excel =====
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", date_format="DD/MM/YYYY") as writer:
    df.to_excel(writer, sheet_name=f"{SHEET_NAME}_processed", index=False)
    extraction.to_excel(writer, sheet_name="Extraction", index=False)

print(f"✔ Terminé : {OUTPUT_FILE}")


In [18]:
# Chercher la colonne email par nom, sinon fallback sur la 9e colonne
cands = [c for c in df_src.columns if c.lower() in {
    "agent_email", "email", "mail", "email_address", "adresse_email"
}]
email_col = cands[0] if cands else df_src.columns[8]  # colonne I (index 8) par défaut
print("Colonne email retenue :", email_col)


Colonne email retenue : EMAIL AGENT


In [19]:
# Résumé: nombre de lignes par email (tri décroissant)
summary = (df_src
           .groupby(email_col, dropna=False)
           .size()
           .reset_index(name="NB_LIGNES")
           .sort_values("NB_LIGNES", ascending=False)
           .reset_index(drop=True))

print("📬 Récapitulatif par email (top 50) :")
display(summary.head(50))

# Aperçu: 3 premières lignes par email
apercu = (df_src
          .assign(_n=df_src.groupby(email_col).cumcount())
          .query("_n < 3")
          .drop(columns=["_n"]))
print("🔎 Aperçu (3 lignes par email) :")
display(apercu)


📬 Récapitulatif par email (top 50) :


,EMAIL AGENT,NB_LIGNES
0,rc@leadway.com,2764
1,koffijpallangba@gmail.com,202
2,zemgbonarcisse@yahoo.fr,193
3,wassassur@gmail.com,190
4,julien7koffi@gmail.com,177
5,assiyapiarmand24@gmail.com,172
6,mohamedbilla2015@gmail.com,171
7,visionassurances@gmail.com,149
8,mevalykone04983342@gmail.com,139
9,gnira.corporate@gmail.com,136


🔎 Aperçu (3 lignes par email) :


,NOM COMPLET,PRODUIT,PRIME TTC,IMMATRICULATION,TELEPHONE,DUREE ( MOIS),DATE ECHEANCE,LIEN DE RENOUVELLEMENT,EMAIL AGENT
0,PEHE SOUEHI JEAN-PAUL,MOTO,8580,4877LT04,0777084048,12.000000,2025-08-20,https://ci.leadway.com/auto/quote/SNE32E342B2024,hamidouzongo92@gmail.com
1,SIELOU HOUSSOU TANOH KOUAME NOE,MOTO,8580,6467JL01,0708703135,12.000000,2025-08-09,https://ci.leadway.com/auto/quote/SN860E64212024,lalayoyo12@yahoo.fr
2,KONAN OI KONAN EVARISTE,MOTO,8580,6485JS04,0779491211,12.000000,2025-08-19,https://ci.leadway.com/auto/quote/SN9F902B2A2024,ozidjezago10@gmail.com
3,BAMBA TILONON DAMIEN,MOTO,8580,8827JV04,0505465065,12.166667,2025-08-06,https://ci.leadway.com/auto/quote/SN426BB4E32024,2005quaddy@gmail.com
4,DIAKITE SIAKA ADAM,SUR_MESURE,82950,8137JE01,0707376230,12.166667,2025-08-23,https://ci.leadway.com/auto/quote/SN8AEE4B7F2024,senfousenfou10@gmail.com
...,...,...,...,...,...,...,...,...,...
19678,Kone inza,TIERS_SIMPLE,12112,418GU01,NaN,30.000000,2025-08-31,https://ci.leadway.com/auto/quote/SN1942068F2025,makourasoutra123@gmail.com
19681,CMEC /PC OUATTARA SOUALIHOU,TIERS_SIMPLE,13302,8188GB01,NaN,30.000000,2025-08-31,https://ci.leadway.com/auto/quote/SN9A4029922025,kamagatemahama4@gmail.com
19747,KABA ABOUBAKAR EL SUDICK,TIERS_SIMPLE,13223,3500ET01,NaN,30.000000,2025-08-31,https://ci.leadway.com/auto/quote/SN67F2CF312025,mariamcoulibaly0000@gmail.com
19750,KONATÉ IBRAHIM KALIL,TIERS_SIMPLE,8923,2707HB01,NaN,30.000000,2025-08-31,https://ci.leadway.com/auto/quote/SNA276F7D22025,koffidjen@gmail.com


In [20]:
import pandas as pd
from pathlib import Path

# ===== Fichiers / feuilles =====
OUTPUT_FILE  = "/Users/y-kouadio/Documents/DATA/global_database_processed.xlsx"  # adapte si besoin
SOURCE_SHEET = "Extraction"
OUT_SHEET    = "Extraction_Grouped"

# ===== Paramètres =====
N_COLS = 9                      # on reprend A..I comme dans le VBA
BLOCK_W = N_COLS + 1            # 1 colonne vide entre blocs
MAX_XLSX_COLS = 16384           # limite Excel

# 1) Lire l'onglet "Extraction"
df_src = pd.read_excel(OUTPUT_FILE, sheet_name=SOURCE_SHEET, engine="openpyxl")
print("Extraction -> shape:", df_src.shape)

# 2) Trouver la colonne email (par nom), sinon prendre la 9e (colonne I)
cands = [c for c in df_src.columns if c.lower() in {"agent_email","email","mail","email_address","adresse_email"}]
email_col = cands[0] if cands else df_src.columns[8]
print("Colonne email retenue:", email_col)

# 3) Regrouper et écrire côte à côte dans 'Extraction_Grouped'
cols_to_copy = list(df_src.columns[:N_COLS])
groups = list(df_src.groupby(email_col, dropna=False))
max_blocks_per_row = max(1, MAX_XLSX_COLS // BLOCK_W)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    # (réécrire Extraction propre si tu veux) : df_src.to_excel(w, sheet_name=SOURCE_SHEET, index=False)
    current_row = 0
    for i in range(0, len(groups), max_blocks_per_row):
        band = groups[i : i + max_blocks_per_row]
        heights = []
        for j, (email, grp) in enumerate(band):
            block = grp[cols_to_copy]
            start_col = j * BLOCK_W  # 0, 10, 20, ...
            block.to_excel(w, sheet_name=OUT_SHEET, index=False,
                           startrow=current_row, startcol=start_col)
            heights.append(len(block) + 1)  # +1 pour l’en-tête
        current_row += (max(heights) if heights else 0) + 1

print(f"✅ Vue groupée écrite dans l’onglet '{OUT_SHEET}' de :", OUTPUT_FILE)

# 4) APERÇU : résumé par email + premières lignes de la vue groupée
# 4a) résumé (nb de lignes par email)
summary = (df_src.groupby(email_col, dropna=False)
                 .size()
                 .reset_index(name="NB_LIGNES")
                 .sort_values("NB_LIGNES", ascending=False)
                 .reset_index(drop=True))
print("\n📬 Récapitulatif par email (top 30) :")
display(summary.head(30))

# 4b) lire la feuille groupée et afficher les 20 premières lignes / 30 premières colonnes
df_grouped_preview = pd.read_excel(OUTPUT_FILE, sheet_name=OUT_SHEET, engine="openpyxl")
print("\n👀 Aperçu de 'Extraction_Grouped' (20 lignes × 30 colonnes max) :")
display(df_grouped_preview.iloc[:20, :min(30, df_grouped_preview.shape[1])])


Extraction -> shape: (19879, 9)
Colonne email retenue: EMAIL AGENT
✅ Vue groupée écrite dans l’onglet 'Extraction_Grouped' de : /Users/y-kouadio/Documents/DATA/global_database_processed.xlsx

📬 Récapitulatif par email (top 30) :


,EMAIL AGENT,NB_LIGNES
0,rc@leadway.com,2764
1,koffijpallangba@gmail.com,202
2,zemgbonarcisse@yahoo.fr,193
3,wassassur@gmail.com,190
4,julien7koffi@gmail.com,177
5,assiyapiarmand24@gmail.com,172
6,mohamedbilla2015@gmail.com,171
7,visionassurances@gmail.com,149
8,mevalykone04983342@gmail.com,139
9,gnira.corporate@gmail.com,136



👀 Aperçu de 'Extraction_Grouped' (20 lignes × 30 colonnes max) :


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# =========================
# ÉTAPE 3 : Excel / PDF / Mail
# =========================
# Prérequis (une seule fois) :
# %pip install pandas openpyxl python-dotenv reportlab

import os, re
import pandas as pd
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# --- PDF (ReportLab)
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Image, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet

# --- Email (SMTP)
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

# ============ PARAMS À ADAPTER ============
OUTPUT_FILE   = "/Users/y-kouadio/Documents/DATA/global_database_processed.xlsx"  # fichier avec l’onglet 'Extraction'
SOURCE_SHEET  = "Extraction"
LOGO_PATH     = "/Users/y-kouadio/Documents/DATA/Image4.jpg"  # ou None si pas de logo
DRY_RUN       = True   # ← mets False pour ENVOYER les e-mails
COLS_TO_SEND  = None   # ex: ["client_id","nom_client","fin_contrat","LIEN DE RENOUVELLEMENT"] ; None = toutes colonnes

# SMTP (.env recommandé)
# .env :
# SMTP_HOST=smtp.office365.com
# SMTP_PORT=587
# SMTP_USER=ton_mail@domaine.com
# SMTP_PASS=********
# FROM_EMAIL=notifications@domaine.com
load_dotenv()
SMTP_HOST   = os.getenv("SMTP_HOST", "smtp.office365.com")
SMTP_PORT   = int(os.getenv("SMTP_PORT", "587"))
SMTP_USER   = os.getenv("SMTP_USER")
SMTP_PASS   = os.getenv("SMTP_PASS")
FROM_EMAIL  = os.getenv("FROM_EMAIL", SMTP_USER or "no-reply@local")

# ============ OUTILS ============

def month_label_next():
    # étiquette "Mois Année" pour le MOIS PROCHAIN
    today = datetime.today()
    first_next = (pd.Timestamp(today.date()).replace(day=1) + pd.offsets.MonthBegin(1))
    return first_next.strftime("%B %Y")  # ex: "September 2025"

def safe_stem(email: str) -> str:
    if not email:
        return "SansEmail"
    stem = email.strip().lower()
    stem = stem.replace("@", "_at_")
    stem = re.sub(r"[^a-z0-9._-]+", "_", stem)
    return stem

def make_pdf(df_agent: pd.DataFrame, pdf_path: Path, title: str = "", logo_path: str | None = None):
    pdf_path = Path(pdf_path)
    styles = getSampleStyleSheet()
    story = []

    # Logo (optionnel)
    if logo_path and Path(logo_path).exists():
        try:
            story.append(Image(logo_path, width=120, height=50))
            story.append(Spacer(1, 8))
        except Exception:
            pass

    # Titre
    if title:
        story.append(Paragraph(title, styles["Title"]))
        story.append(Spacer(1, 8))

    # Tableau (entête orange + grille fine)
    data = [df_agent.columns.tolist()] + df_agent.fillna("").astype(str).values.tolist()
    table = Table(data, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.orange),
        ("TEXTCOLOR",  (0,0), (-1,0), colors.black),
        ("GRID",       (0,0), (-1,-1), 0.25, colors.black),
        ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
        ("FONTSIZE",   (0,0), (-1,-1), 9),
        ("ALIGN",      (0,0), (-1,0), "CENTER"),
        ("VALIGN",     (0,0), (-1,-1), "MIDDLE"),
    ]))
    doc = SimpleDocTemplate(str(pdf_path), pagesize=A4, leftMargin=24, rightMargin=24, topMargin=24, bottomMargin=24)
    story.append(table)
    doc.build(story)

def send_email_with_attachments(to_addr: str, subject: str, body: str, files: list[Path]):
    msg = MIMEMultipart()
    msg["Subject"] = subject
    msg["From"]    = FROM_EMAIL
    msg["To"]      = to_addr
    msg.attach(MIMEText(body, "plain", "utf-8"))

    for fp in files:
        fp = Path(fp)
        if not fp.exists():
            continue
        with open(fp, "rb") as f:
            part = MIMEApplication(f.read(), Name=fp.name)
        part["Content-Disposition"] = f'attachment; filename="{fp.name}"'
        msg.attach(part)

    with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
        server.starttls()
        if SMTP_USER and SMTP_PASS:
            server.login(SMTP_USER, SMTP_PASS)
        server.send_message(msg)

# ============ 1) LIRE L’ONGLET EXTRACTION ============
# Lecture robuste : openpyxl puis fallback calamine si besoin
try:
    df_src = pd.read_excel(OUTPUT_FILE, sheet_name=SOURCE_SHEET, engine="openpyxl")
except Exception:
    %pip install -q python-calamine
    df_src = pd.read_excel(OUTPUT_FILE, sheet_name=SOURCE_SHEET, engine="calamine")

# Détermination de la colonne email
cands = [c for c in df_src.columns if c.lower() in {"agent_email","email","mail","email_address","adresse_email"}]
email_col = cands[0] if cands else df_src.columns[8]  # fallback colonne I
cols_finales = (COLS_TO_SEND if COLS_TO_SEND else list(df_src.columns))

# ============ 2) DÉFINIR LE DOSSIER DE SORTIE ============
base_dir = Path(OUTPUT_FILE).parent
batch_label = datetime.today().strftime("renouvellements_%Y%m")
out_dir = base_dir / batch_label
out_dir.mkdir(parents=True, exist_ok=True)
print("Dossier de sortie :", out_dir)

# ============ 3) BOUCLE PAR AGENT : Excel + PDF ============
groups = list(df_src.groupby(email_col, dropna=False))
print("Nombre d'agents détectés :", len(groups))

export_log = []  # pour un récap

for email, grp in groups:
    # ignorer les emails vides
    if pd.isna(email) or str(email).strip() == "":
        continue

    df_agent = grp[cols_finales].copy()
    stem = safe_stem(str(email))
    xlsx_path = out_dir / f"renouvellements_{stem}.xlsx"
    pdf_path  = out_dir / f"renouvellements_{stem}.pdf"

    # Excel
    with pd.ExcelWriter(xlsx_path, engine="openpyxl", datetime_format="DD/MM/YYYY", date_format="DD/MM/YYYY") as w:
        df_agent.to_excel(w, sheet_name="Renouvellements", index=False)

    # PDF
    titre = f"Contrats à renouveler – {month_label_next()} – {email}"
    make_pdf(df_agent, pdf_path, title=titre, logo_path=LOGO_PATH)

    export_log.append({"email": email, "rows": len(df_agent), "excel": str(xlsx_path), "pdf": str(pdf_path)})

print("Exports générés pour", len(export_log), "agents.")

# ============ 4) ENVOI DES MAILS ============
subject = f"Renouvellements à échéance – {month_label_next()}"
body = (
    "Bonjour,\n\n"
    "Vous trouverez en pièce jointe la liste des contrats de vos clients à renouveler.\n"
    "Merci d’utiliser le « LIEN DE RENOUVELLEMENT » présent dans le fichier pour traiter chaque dossier.\n\n"
    "— Équipe Assurance"
)

if DRY_RUN:
    print("DRY_RUN=True → AUCUN MAIL ENVOYÉ. (Mets DRY_RUN=False pour envoyer)")
else:
    sent = 0
    for row in export_log:
        try:
            send_email_with_attachments(
                to_addr=row["email"],
                subject=subject,
                body=body,
                files=[row["excel"], row["pdf"]]
            )
            sent += 1
        except Exception as e:
            print(f"[!] Échec envoi à {row['email']} → {e}")
    print(f"✅ Emails envoyés : {sent}/{len(export_log)}")

# ============ 5) APERÇU / RÉCAP ============
recap = pd.DataFrame(export_log).sort_values("rows", ascending=False).reset_index(drop=True)
print("\n📬 Récapitulatif (top 20 par volume) :")
display(recap.head(20))
print("\nTous les fichiers sont dans :", out_dir)


Dossier de sortie : /Users/y-kouadio/Documents/DATA/renouvellements_202508
Nombre d'agents détectés : 1126
Exports générés pour 1126 agents.
DRY_RUN=True → AUCUN MAIL ENVOYÉ. (Mets DRY_RUN=False pour envoyer)

📬 Récapitulatif (top 20 par volume) :


,email,rows,excel,pdf
0,rc@leadway.com,2764,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
1,koffijpallangba@gmail.com,202,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
2,zemgbonarcisse@yahoo.fr,193,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
3,wassassur@gmail.com,190,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
4,julien7koffi@gmail.com,177,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
5,assiyapiarmand24@gmail.com,172,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
6,mohamedbilla2015@gmail.com,171,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
7,visionassurances@gmail.com,149,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
8,mevalykone04983342@gmail.com,139,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...
9,gnira.corporate@gmail.com,136,/Users/y-kouadio/Documents/DATA/renouvellement...,/Users/y-kouadio/Documents/DATA/renouvellement...



Tous les fichiers sont dans : /Users/y-kouadio/Documents/DATA/renouvellements_202508


In [22]:
%pip install -U reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 774.8 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [23]:
import reportlab
print("ReportLab:", reportlab.Version)


ReportLab: 4.4.3


In [27]:
DRY_RUN = False
ALLOW_LIST = {
    "y-kouadio@leadway.com",   # ← mets la tienne
    "yarakouadio@gmail.com",
}

sent = 0; skipped = 0
for row in export_log:
    to_ = str(row["email"]).strip().lower()
    if to_ not in ALLOW_LIST:
        skipped += 1
        continue
    try:
        send_email_with_attachments(
            to_addr=to_,
            subject=f"Renouvellements à échéance – {month_label_next()}",
            body=("Bonjour,\n\n"
                  "Vous trouverez en pièce jointe la liste des contrats à renouveler.\n"
                  "Utilisez le « LIEN DE RENOUVELLEMENT » dans le fichier.\n\n"
                  "— Équipe Assurance"),
            files=[row["excel"], row["pdf"]]
        )
        sent += 1
        print("OK →", to_)
    except Exception as e:
        print("ERREUR →", to_, ":", e)

print(f"Envoyés: {sent} | ignorés (hors ALLOW_LIST): {skipped}")


Envoyés: 0 | ignorés (hors ALLOW_LIST): 1126


In [28]:
# prend le premier paquet généré
test_pkg = export_log[0]
print(test_pkg)

TEST_RECIPIENTS = ["y-kouadio@leadway.com", "yarakouadio@gmail.com"]  # tes adresses
for to_ in TEST_RECIPIENTS:
    try:
        send_email_with_attachments(
            to_addr=to_.strip(),
            subject=f"TEST – Renouvellements à échéance – {month_label_next()}",
            body=("Bonjour,\n\n"
                  "Ceci est un TEST d'envoi. "
                  "Pièces jointes: 1 Excel + 1 PDF d'un agent aléatoire.\n\n"
                  "— Équipe Assurance"),
            files=[test_pkg["excel"], test_pkg["pdf"]]
        )
        print("OK →", to_)
    except Exception as e:
        print("ERREUR →", to_, ":", e)


{'email': '2005quaddy@gmail.com', 'rows': 5, 'excel': '/Users/y-kouadio/Documents/DATA/renouvellements_202508/renouvellements_2005quaddy_at_gmail.com.xlsx', 'pdf': '/Users/y-kouadio/Documents/DATA/renouvellements_202508/renouvellements_2005quaddy_at_gmail.com.pdf'}
ERREUR → y-kouadio@leadway.com : (530, b'5.7.57 Client not authenticated to send mail. [CT2P275CA0037.ZAFP275.PROD.OUTLOOK.COM 2025-08-10T02:28:36.103Z 08DDD771BC2BE757]', 'no-reply@local')
ERREUR → yarakouadio@gmail.com : (530, b'5.7.57 Client not authenticated to send mail. [CTXP275CA0048.ZAFP275.PROD.OUTLOOK.COM 2025-08-10T02:28:42.563Z 08DDD77D9374CE4D]', 'no-reply@local')


In [30]:
# --- Credentials SMTP (en clair pour tester rapidement) ---
SMTP_HOST = "smtp.office365.com"
SMTP_PORT = 587
SMTP_USER = "y-kouadio@leadway.com"     # ton adresse M365
SMTP_PASS = "TON_MOT_DE_PASSE_OU_MDP_D_APPLI"  # si MFA, utilise un mot de passe d’application
FROM_EMAIL = SMTP_USER                   # garde la même adresse comme expéditeur



In [33]:
SMTP_HOST="smtp.gmail.com"; SMTP_PORT=587
SMTP_USER="yaraehounou@gmail.com"; SMTP_PASS="MDP_APP"


In [35]:
%pip install sendgrid
import os
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Attachment, FileContent, FileName, FileType, Disposition

sg = SendGridAPIClient("SG.XXX_TA_CLE_API_XXX")
message = Mail(
    from_email="notifications@tondomaine.com",
    to_emails="yaraehounou@gmail.com",
    subject="TEST SendGrid",
    plain_text_content="Test pièces jointes.")
# attacher un fichier si tu veux …
sg.send(message)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sendgrid]
Note: you may need to restart the kernel to use updated packages.


UnauthorizedError: HTTP Error 401: Unauthorized

In [36]:
import os
from dotenv import load_dotenv
load_dotenv()  # si tu utilises un .env

SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY")  # OU mets-la en dur: "SG.xxxxx..."
print("Clé présente ?", bool(SENDGRID_API_KEY))
if SENDGRID_API_KEY:
    print("Préfixe:", SENDGRID_API_KEY[:3], "| longueur:", len(SENDGRID_API_KEY))
    SENDGRID_API_KEY = SENDGRID_API_KEY.strip()


Clé présente ? False


In [37]:
import os, pathlib
print("Dossier courant :", os.getcwd())
print("Contenu :", [p.name for p in pathlib.Path('.').iterdir()])


Dossier courant : /Users/y-kouadio/ENVOI DE MAIL AUTOMATIQUE
Contenu : ['Untitled.ipynb', 'global_database_processed.xlsx', '.ipynb_checkpoints', 'ANALYSE LEADWAY']
